# Gold Layer - Star Schema for Analytics (Unity Catalog Compatible)

**Source:** `workspace.rebu.silver_*` tables  
**Target:** `workspace.rebu.gold_*` tables

This notebook creates a Star Schema in the Gold layer optimized for analytics and dashboards.

---

## Summary of Steps

1. **Setup and Configuration:** Initialize Spark session and Unity Catalog settings.
2. **Helper Functions:** Define functions for table naming and Gold table creation.
3. **Verify Silver Tables:** Check existence of required Silver layer tables.
4. **Star Schema Design:** Outline dimension and fact tables for analytics.
5. **Create Dimension Tables:** Build and save driver, passenger, taxi, location, payment, date, and time dimensions.
6. **Create Fact Tables:** Construct and save booking and rewards fact tables.
7. **Save to Gold Layer:** Write all tables to Unity Catalog Gold layer for downstream analytics.

## Setup and Configuration

This section initializes the Spark session and sets up Unity Catalog configuration variables.  
It prepares the environment for reading Silver layer tables and writing Gold layer tables, ensuring all subsequent operations use the correct catalog and schema.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, when, lit, concat, concat_ws, year, month, dayofmonth, 
    hour, dayofweek, date_format, datediff, round as spark_round, 
    row_number, dense_rank, current_timestamp, sum as spark_sum, 
    avg, count, max, min
)
from delta.tables import DeltaTable

# Unity Catalog Configuration
UC_CATALOG = "workspace"
UC_SCHEMA = "rebu"

print("="*80)
print("GOLD LAYER CONFIGURATION - STAR SCHEMA")
print("="*80)
print(f"Reading from: {UC_CATALOG}.{UC_SCHEMA}.silver_*")
print(f"Writing to: {UC_CATALOG}.{UC_SCHEMA}.gold_*")
print("="*80)

## Helper Functions

**Cell 5:**  
Defines two functions to generate fully qualified table names for Silver and Gold layers using Unity Catalog variables (`UC_CATALOG`, `UC_SCHEMA`).  
- `silver_table(name)`: Returns the full Silver table name (e.g., `workspace.rebu.silver_drivers`).  
- `gold_table(name)`: Returns the full Gold table name (e.g., `workspace.rebu.gold_drivers`).

**Cell 6:**  
Defines `create_gold_table(df, table_name, description="")` to save a DataFrame as a Gold layer Delta table:  
- Adds a `gold_created_timestamp` column for metadata.  
- Writes the DataFrame to the Gold layer table with overwrite mode and schema update.  
- Prints creation status and record count.  
- Returns the DataFrame with metadata.

In [0]:
def silver_table(name):
    """Get full Silver table name"""
    return f"{UC_CATALOG}.{UC_SCHEMA}.silver_{name}"

def gold_table(name):
    """Get full Gold table name"""
    return f"{UC_CATALOG}.{UC_SCHEMA}.gold_{name}"

In [0]:
def create_gold_table(df, table_name, description=""):
    """Save dataframe as Gold layer Delta table"""
    
    print(f"\n{'='*80}")
    print(f"Creating: gold_{table_name}")
    if description:
        print(f"Description: {description}")
    print(f"{'='*80}")
    
    # Add Gold layer metadata
    df_with_metadata = df.withColumn("gold_created_timestamp", current_timestamp())
    
    # Write to Gold layer
    gold_table_full = gold_table(table_name)
    print(f"Target: {gold_table_full}")
    
    df_with_metadata.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(gold_table_full)
    
    record_count = df_with_metadata.count()
    print(f"✓ SUCCESS: Created with {record_count:,} records")
    print(f"{'='*80}\n")
    
    return df_with_metadata

## Verify Silver Tables Exist

This section checks if all required Silver layer tables are available in Unity Catalog.  
It attempts to read each table and prints the record count or a "NOT FOUND" message if missing.  
Missing tables are listed, and instructions are provided to run the Silver layer notebook if needed.

In [0]:
print("\n Checking Silver tables...")
print("-"*80)

silver_tables = [
    "taxi_type", "taxi", "drivers", "passengers", 
    "payment_type", "place", "booking", "rewards"
]

missing_tables = []
for table in silver_tables:
    table_full = silver_table(table)
    try:
        count = spark.table(table_full).count()
        print(f"{table_full:<55} {count:>10,} records")
    except Exception as e:
        print(f"{table_full:<55} NOT FOUND")
        missing_tables.append(table)

if missing_tables:
    print("\n Some Silver tables are missing!")
    print("   Please run the Silver layer notebook first:")
    print("   02_Silver_Layer_UC.py")
    print(f"\n   Missing tables: {', '.join(missing_tables)}")
else:
    print("\n All Silver tables found!")

print("-"*80)

# Star Schema Design

## Dimension Tables
1. **dim_driver** - Driver information
2. **dim_passenger** - Passenger information  
3. **dim_taxi** - Taxi and taxi type information (denormalized)
4. **dim_location** - Pickup and dropoff locations
5. **dim_payment** - Payment type information
6. **dim_date** - Date dimension for time-based analysis
7. **dim_time** - Time dimension for intraday analysis

## Fact Tables
1. **fact_booking** - Booking transactions (main fact table)
2. **fact_rewards** - Rewards accumulated by passengers

## Create Dimension Tables

- **dim_driver:** Builds driver dimension with experience and rating categories.
- **dim_passenger:** Creates passenger dimension with age groups and membership tiers.
- **dim_taxi:** Joins taxi and taxi type tables for a denormalized taxi dimension.
- **dim_location:** Maps places to Singapore geographical zones for location dimension.
- **dim_payment:** Categorizes payment types (Cash, Card, Digital Wallet, Other).
- **dim_date:** Generates a date dimension from booking, pickup, and dropoff dates.
- **dim_time:** Constructs a time dimension for 24 hours with labels and time periods.

### Dimension 1: Driver

In [0]:
# Read from Silver layer with full UC name
df_drivers = spark.table(silver_table("drivers"))

# Create dimension with business logic
dim_driver = df_drivers.select(
    col("DriverID").alias("driver_key"),
    col("FirstName").alias("driver_first_name"),
    col("LastName").alias("driver_last_name"),
    concat_ws(" ", col("FirstName"), col("LastName")).alias("driver_full_name"),
    col("Phone").alias("driver_phone"),
    col("Email").alias("driver_email"),
    col("LicenseNumber").alias("driver_license_number"),
    col("Age").alias("driver_age"),
    col("Gender").alias("driver_gender"),
    col("YearsExperience").alias("driver_years_experience"),
    col("Rating").alias("driver_rating"),
    col("JoinDate").alias("driver_join_date"),
    # Add derived attributes
    when(col("YearsExperience") < 2, "Novice")
    .when(col("YearsExperience").between(2, 5), "Intermediate")
    .when(col("YearsExperience") > 5, "Expert")
    .otherwise("Unknown").alias("driver_experience_level"),
    when(col("Rating") >= 4.5, "Excellent")
    .when(col("Rating") >= 4.0, "Good")
    .when(col("Rating") >= 3.5, "Average")
    .when(col("Rating") < 3.5, "Below Average")
    .otherwise("Not Rated").alias("driver_rating_category")
)

gold_dim_driver = create_gold_table(
    dim_driver, 
    "dim_driver",
    "Driver dimension with experience and rating categories"
)
display(gold_dim_driver.limit(10))

### Dimension 2: Passenger

In [0]:
df_passengers = spark.table(silver_table("passengers"))

dim_passenger = df_passengers.select(
    col("PassengerID").alias("passenger_key"),
    col("FirstName").alias("passenger_first_name"),
    col("LastName").alias("passenger_last_name"),
    concat_ws(" ", col("FirstName"), col("LastName")).alias("passenger_full_name"),
    col("Phone").alias("passenger_phone"),
    col("Email").alias("passenger_email"),
    col("MemberCategory").alias("passenger_member_category"),
    col("Age").alias("passenger_age"),
    col("Gender").alias("passenger_gender"),
    col("Address").alias("passenger_address"),
    col("RegistrationDate").alias("passenger_registration_date"),
    # Add derived attributes
    when(col("Age") < 25, "18-24")
    .when(col("Age").between(25, 34), "25-34")
    .when(col("Age").between(35, 44), "35-44")
    .when(col("Age").between(45, 54), "45-54")
    .when(col("Age").between(55, 64), "55-64")
    .otherwise("65+").alias("passenger_age_group"),
    when(col("MemberCategory") == "A", "Premium")
    .when(col("MemberCategory") == "B", "Standard")
    .when(col("MemberCategory") == "C", "Basic")
    .otherwise("Unknown").alias("passenger_membership_tier")
)

gold_dim_passenger = create_gold_table(
    dim_passenger,
    "dim_passenger",
    "Passenger dimension with age groups and membership tiers"
)
display(gold_dim_passenger.limit(10))

### Dimension 3: Taxi (Denormalized with Taxi Type)

In [0]:
df_taxi = spark.table(silver_table("taxi"))
df_taxi_type = spark.table(silver_table("taxi_type"))

# Join Taxi with TaxiType for denormalization
dim_taxi = df_taxi.join(
    df_taxi_type,
    df_taxi.TaxiTypeID == df_taxi_type.TaxiTypeID,
    "left"
).select(
    col("TaxiID").alias("taxi_key"),
    col("LicensePlate").alias("taxi_license_plate"),
    col("Model").alias("taxi_model"),
    col("YearManufactured").alias("taxi_year"),
    col("Status").alias("taxi_status"),
    df_taxi_type.TaxiTypeID.alias("taxi_type_id"),
    df_taxi_type.TypeName.alias("taxi_type_name"),
    df_taxi_type.BaseFare.alias("taxi_base_fare"),
    df_taxi_type.PerKmRate.alias("taxi_per_km_rate"),
    df_taxi_type.Capacity.alias("taxi_capacity"),
    # Add derived attributes
    (year(current_timestamp()) - col("YearManufactured")).alias("taxi_age_years"),
    when(col("Status") == "Active", True).otherwise(False).alias("taxi_is_active")
)

gold_dim_taxi = create_gold_table(
    dim_taxi,
    "dim_taxi",
    "Taxi dimension denormalized with taxi type details"
)
display(gold_dim_taxi.limit(10))

### Dimension 4: Location (Place)

In [0]:
df_place = spark.table(silver_table("place"))

dim_location = df_place.select(
    col("PlaceID").alias("location_key"),
    col("PlaceName").alias("location_name"),
    col("Latitude").alias("location_latitude"),
    col("Longitude").alias("location_longitude"),
    col("PostalCode").alias("location_postal_code"),
    # Add derived attributes for geographical zones (Singapore specific)
    when(col("PlaceName").contains("Orchard"), "Central")
    .when(col("PlaceName").contains("Marina"), "Central")
    .when(col("PlaceName").contains("Raffles"), "Central")
    .when(col("PlaceName").contains("Clarke"), "Central")
    .when(col("PlaceName").contains("Changi"), "East")
    .when(col("PlaceName").contains("Tampines"), "East")
    .when(col("PlaceName").contains("Pasir Ris"), "East")
    .when(col("PlaceName").contains("Jurong"), "West")
    .when(col("PlaceName").contains("Clementi"), "West")
    .when(col("PlaceName").contains("Bukit Batok"), "West")
    .when(col("PlaceName").contains("Ang Mo Kio"), "North")
    .when(col("PlaceName").contains("Yishun"), "North")
    .when(col("PlaceName").contains("Sembawang"), "North")
    .otherwise("Other").alias("location_zone")
)

gold_dim_location = create_gold_table(
    dim_location,
    "dim_location",
    "Location dimension with Singapore geographical zones"
)
display(gold_dim_location.limit(10))

### Dimension 5: Payment Type

In [0]:
df_payment_type = spark.table(silver_table("payment_type"))

dim_payment = df_payment_type.select(
    col("PaymentTypeID").alias("payment_key"),
    col("TypeName").alias("payment_type_name"),
    # Add derived attributes
    when(col("TypeName").isin("Cash"), "Cash")
    .when(col("TypeName").isin("Credit Card", "Debit Card"), "Card")
    .when(col("TypeName").isin("PayNow"), "Digital Wallet")
    .otherwise("Other").alias("payment_category")
)

gold_dim_payment = create_gold_table(
    dim_payment,
    "dim_payment",
    "Payment type dimension with payment categories"
)
display(gold_dim_payment)

### Dimension 6: Date

In [0]:
# Generate date dimension from booking data
df_booking = spark.table(silver_table("booking"))

# Get date range from bookings
date_range = df_booking.select(
    col("BookingDateTime").alias("date_val")
).union(
    df_booking.select(col("PickupTime").alias("date_val"))
).union(
    df_booking.select(col("DropoffTime").alias("date_val"))
)

# Create comprehensive date dimension
dim_date = date_range.select(
    col("date_val").cast("date").alias("date_key")
).distinct().select(
    col("date_key"),
    year(col("date_key")).alias("year"),
    month(col("date_key")).alias("month"),
    dayofmonth(col("date_key")).alias("day"),
    dayofweek(col("date_key")).alias("day_of_week"),
    date_format(col("date_key"), "EEEE").alias("day_name"),
    date_format(col("date_key"), "MMMM").alias("month_name"),
    concat(year(col("date_key")), lit("-Q"), 
           when(month(col("date_key")).between(1, 3), 1)
           .when(month(col("date_key")).between(4, 6), 2)
           .when(month(col("date_key")).between(7, 9), 3)
           .otherwise(4)).alias("quarter"),
    when(dayofweek(col("date_key")).isin(1, 7), True).otherwise(False).alias("is_weekend"),
    when(dayofweek(col("date_key")).between(2, 6), True).otherwise(False).alias("is_weekday")
).orderBy("date_key")

gold_dim_date = create_gold_table(
    dim_date,
    "dim_date",
    "Date dimension for time-based analysis"
)
display(gold_dim_date.limit(10))

### Dimension 7: Time

In [0]:
# Create time dimension (24 hours)
from pyspark.sql.types import StructType, StructField, IntegerType

# Generate hours 0-23
hours_data = [(h,) for h in range(24)]
hours_schema = StructType([StructField("hour", IntegerType(), False)])
df_hours = spark.createDataFrame(hours_data, hours_schema)

dim_time = df_hours.select(
    col("hour").alias("time_key"),
    col("hour").alias("hour_of_day"),
    when(col("hour") == 0, "12 AM")
    .when(col("hour") < 12, concat(col("hour"), lit(" AM")))
    .when(col("hour") == 12, "12 PM")
    .otherwise(concat(col("hour") - 12, lit(" PM"))).alias("hour_label"),
    # Time periods
    when(col("hour").between(6, 11), "Morning")
    .when(col("hour").between(12, 17), "Afternoon")
    .when(col("hour").between(18, 21), "Evening")
    .otherwise("Night").alias("time_period"),
    when(col("hour").between(7, 9), True)
    .when(col("hour").between(17, 19), True)
    .otherwise(False).alias("is_peak_hour"),
    when(col("hour").between(22, 5), True).otherwise(False).alias("is_late_night")
)

gold_dim_time = create_gold_table(
    dim_time,
    "dim_time",
    "Time dimension for intraday analysis"
)
display(gold_dim_time)

## Create Fact Tables


- Builds the **fact_booking** table from Silver booking data.
- Adds dimension keys, timestamps, trip measures (distance, fare, duration, wait time), and status flags.
- Saves as Gold layer table for booking transactions.
- Builds the **fact_rewards** table from Silver rewards data.
- Includes reward keys, passenger keys, earned date, points, description, and reward tier.
- Saves as Gold layer table for passenger rewards tracking.

### Fact 1: Booking (Main Fact Table)

In [0]:
df_booking = spark.table(silver_table("booking"))

# Create fact table with all foreign keys and measures
fact_booking = df_booking.select(
    col("BookingID").alias("booking_key"),
    # Foreign Keys to Dimensions
    col("PassengerID").alias("passenger_key"),
    col("DriverID").alias("driver_key"),
    col("TaxiID").alias("taxi_key"),
    col("PickupPlaceID").alias("pickup_location_key"),
    col("DropoffPlaceID").alias("dropoff_location_key"),
    # Date/Time Keys
    col("BookingDateTime").cast("date").alias("booking_date_key"),
    hour(col("BookingDateTime")).alias("booking_time_key"),
    col("PickupTime").cast("date").alias("pickup_date_key"),
    hour(col("PickupTime")).alias("pickup_time_key"),
    col("DropoffTime").cast("date").alias("dropoff_date_key"),
    hour(col("DropoffTime")).alias("dropoff_time_key"),
    # Degenerate Dimensions
    col("Status").alias("booking_status"),
    # Timestamps
    col("BookingDateTime").alias("booking_datetime"),
    col("PickupTime").alias("pickup_datetime"),
    col("DropoffTime").alias("dropoff_datetime"),
    # Measures
    col("Distance").alias("trip_distance_km"),
    col("Fare").alias("trip_fare"),
    # Derived Measures - Handle division by zero
    when(col("Distance") > 0, 
         spark_round((col("Fare") / col("Distance")), 2)
    ).otherwise(None).alias("fare_per_km"),
    spark_round(
        (col("DropoffTime").cast("long") - col("PickupTime").cast("long")) / 60.0,
        2
    ).alias("trip_duration_minutes"),
    spark_round(
        (col("PickupTime").cast("long") - col("BookingDateTime").cast("long")) / 60.0,
        2
    ).alias("wait_time_minutes"),
    # Flags
    when(col("Status") == "Completed", 1).otherwise(0).alias("is_completed"),
    when(col("Status") == "Cancelled", 1).otherwise(0).alias("is_cancelled")
)

gold_fact_booking = create_gold_table(
    fact_booking,
    "fact_booking",
    "Main fact table for booking transactions with all measures"
)
display(gold_fact_booking.limit(10))

### Fact 2: Rewards

In [0]:
df_rewards = spark.table(silver_table("rewards"))

fact_rewards = df_rewards.select(
    col("RewardID").alias("reward_key"),
    # Foreign Keys
    col("PassengerID").alias("passenger_key"),
    col("EarnedDate").alias("earned_date_key"),
    # Measures
    col("Points").alias("reward_points"),
    col("Description").alias("reward_description"),
    # Derived attributes
    when(col("Points") < 20, "Low")
    .when(col("Points").between(20, 50), "Medium")
    .when(col("Points") > 50, "High")
    .otherwise("Unknown").alias("reward_tier")
)

gold_fact_rewards = create_gold_table(
    fact_rewards,
    "fact_rewards",
    "Rewards fact table tracking points earned by passengers"
)
display(gold_fact_rewards.limit(10))

## Create Aggregated Fact Tables

- **fact_daily_booking_summary:** Aggregates daily booking metrics (total bookings, revenue, trip stats).
- **fact_driver_performance:** Summarizes driver performance per day (bookings, revenue, trip stats).
- **fact_passenger_activity:** Tracks passenger activity and spending (total bookings, distance, last booking).

### Daily Booking Summary

In [0]:
from pyspark.sql.functions import count as spark_count

daily_summary = spark.table(gold_table("fact_booking")).groupBy(
    col("booking_date_key")
).agg(
    spark_count("*").alias("total_bookings"),
    spark_sum("is_completed").alias("completed_bookings"),
    spark_sum("is_cancelled").alias("cancelled_bookings"),
    spark_sum("trip_fare").alias("total_revenue"),
    avg("trip_fare").alias("avg_fare"),
    spark_sum("trip_distance_km").alias("total_distance_km"),
    avg("trip_distance_km").alias("avg_distance_km"),
    avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
    avg("wait_time_minutes").alias("avg_wait_time_minutes")
)

gold_daily_summary = create_gold_table(
    daily_summary,
    "fact_daily_booking_summary",
    "Daily aggregated booking metrics"
)
display(gold_daily_summary.orderBy(col("booking_date_key").desc()).limit(10))

### Driver Performance Summary

In [0]:
from pyspark.sql.functions import count as spark_count

daily_summary = spark.table(gold_table("fact_booking")).groupBy(
    col("booking_date_key")
).agg(
    spark_count("*").alias("total_bookings"),
    spark_sum("is_completed").alias("completed_bookings"),
    spark_sum("is_cancelled").alias("cancelled_bookings"),
    spark_sum("trip_fare").alias("total_revenue"),
    avg("trip_fare").alias("avg_fare"),
    spark_sum("trip_distance_km").alias("total_distance_km"),
    avg("trip_distance_km").alias("avg_distance_km"),
    avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
    avg("wait_time_minutes").alias("avg_wait_time_minutes")
)

gold_daily_summary = create_gold_table(
    daily_summary,
    "fact_daily_booking_summary",
    "Daily aggregated booking metrics"
)
display(gold_daily_summary.orderBy(col("booking_date_key").desc()).limit(10))

### Passenger Activity Summary

In [0]:
from pyspark.sql.functions import count as spark_count

passenger_activity = spark.table(gold_table("fact_booking")).groupBy(
    col("passenger_key")
).agg(
    spark_count("booking_key").alias("total_bookings"),
    spark_sum("is_completed").alias("completed_bookings"),
    spark_sum("trip_fare").alias("total_spent"),
    avg("trip_fare").alias("avg_fare_per_booking"),
    spark_sum("trip_distance_km").alias("total_distance_traveled"),
    max("booking_datetime").alias("last_booking_date")
)

gold_passenger_activity = create_gold_table(
    passenger_activity,
    "fact_passenger_activity",
    "Passenger activity and spending metrics"
)
display(gold_passenger_activity.orderBy(col("total_spent").desc()).limit(10))

## Data Validation

- Validates star schema by counting records in all dimension and fact tables.
- Prints record counts for each table; reports errors if any table is missing or inaccessible.

In [0]:
print("="*80)
print("STAR SCHEMA VALIDATION")
print("="*80)

# Count records in all tables
dimensions = [
    "dim_driver", "dim_passenger", "dim_taxi", "dim_location", 
    "dim_payment", "dim_date", "dim_time"
]

facts = [
    "fact_booking", "fact_rewards", 
    "fact_daily_booking_summary", "fact_driver_performance", "fact_passenger_activity"
]

print("\nDIMENSION TABLES:")
for dim in dimensions:
    table_full = gold_table(dim)
    try:
        count = spark.table(table_full).count()
        print(f"  {table_full:<60} {count:>10,} records")
    except:
        print(f"  {table_full:<60} {'ERROR':>10}")

print("\nFACT TABLES:")
for fact in facts:
    table_full = gold_table(fact)
    try:
        count = spark.table(table_full).count()
        print(f"  {table_full:<60} {count:>10,} records")
    except:
        print(f"  {table_full:<60} {'ERROR':>10}")

print("="*80)

## Show All Gold Tables

In [0]:
print(f"\nGold Tables in {UC_CATALOG}.{UC_SCHEMA}:")
spark.sql(f"SHOW TABLES IN {UC_CATALOG}.{UC_SCHEMA}") \
    .filter("tableName LIKE 'gold_%'") \
    .select("database", "tableName", "isTemporary") \
    .show(truncate=False)

## Sample Dashboard Queries

### Query 1: Revenue by Taxi Type

In [0]:
spark.sql(f"""
    SELECT 
        t.taxi_type_name,
        COUNT(f.booking_key) as total_trips,
        ROUND(SUM(f.trip_fare), 2) as total_revenue,
        ROUND(AVG(f.trip_fare), 2) as avg_fare,
        ROUND(SUM(f.trip_distance_km), 2) as total_distance_km
    FROM {gold_table('fact_booking')} f
    JOIN {gold_table('dim_taxi')} t ON f.taxi_key = t.taxi_key
    WHERE f.is_completed = 1
    GROUP BY t.taxi_type_name
    ORDER BY total_revenue DESC
""").display()

### Query 2: Peak Hours Analysis

In [0]:
spark.sql(f"""
    SELECT 
        tm.hour_of_day,
        tm.time_period,
        tm.is_peak_hour,
        COUNT(f.booking_key) as total_bookings,
        ROUND(AVG(f.trip_fare), 2) as avg_fare,
        ROUND(AVG(f.trip_duration_minutes), 2) as avg_duration_min
    FROM {gold_table('fact_booking')} f
    JOIN {gold_table('dim_time')} tm ON f.booking_time_key = tm.time_key
    WHERE f.is_completed = 1
    GROUP BY tm.hour_of_day, tm.time_period, tm.is_peak_hour
    ORDER BY tm.hour_of_day
""").display()

### Query 3: Top Routes

In [0]:
spark.sql(f"""
    SELECT 
        pickup.location_name as pickup_location,
        pickup.location_zone as pickup_zone,
        dropoff.location_name as dropoff_location,
        dropoff.location_zone as dropoff_zone,
        COUNT(f.booking_key) as trip_count,
        ROUND(AVG(f.trip_distance_km), 2) as avg_distance_km,
        ROUND(AVG(f.trip_fare), 2) as avg_fare
    FROM {gold_table('fact_booking')} f
    JOIN {gold_table('dim_location')} pickup ON f.pickup_location_key = pickup.location_key
    JOIN {gold_table('dim_location')} dropoff ON f.dropoff_location_key = dropoff.location_key
    WHERE f.is_completed = 1
    GROUP BY pickup.location_name, pickup.location_zone, 
             dropoff.location_name, dropoff.location_zone
    ORDER BY trip_count DESC
    LIMIT 10
""").display()

### Query 4: Monthly Trend

In [0]:
spark.sql(f"""
    SELECT 
        d.year,
        d.month,
        d.month_name,
        d.quarter,
        COUNT(f.booking_key) as total_bookings,
        ROUND(SUM(f.trip_fare), 2) as total_revenue,
        ROUND(AVG(f.trip_fare), 2) as avg_fare
    FROM {gold_table('fact_booking')} f
    JOIN {gold_table('dim_date')} d ON f.booking_date_key = d.date_key
    WHERE f.is_completed = 1
    GROUP BY d.year, d.month, d.month_name, d.quarter
    ORDER BY d.year, d.month
""").display()

## End of Gold Layer Notebook

**Star Schema Components Created:**

**Dimension Tables (7):**
- dim_driver
- dim_passenger
- dim_taxi
- dim_location
- dim_payment
- dim_date
- dim_time

**Fact Tables (5):**
- fact_booking
- fact_rewards
- fact_daily_booking_summary
- fact_driver_performance
- fact_passenger_activity

**Ready for Analytics and BI Dashboards!**